In [ ]:
%pip install -U kagglehub huggingface_hub hf_transfer

In [ ]:
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
KAGGLE_TOKEN = userdata.get("KAGGLE_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("Missing Colab secret: HF_TOKEN")

if not KAGGLE_TOKEN:
    raise RuntimeError("Missing Colab secret: KAGGLE_TOKEN")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["KAGGLE_API_TOKEN"] = KAGGLE_TOKEN
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["DO_NOT_TRACK"] = "1"

print("Secrets loaded: HF_TOKEN -> HF_TOKEN, KAGGLE_TOKEN -> KAGGLE_API_TOKEN")

In [ ]:
from pathlib import Path

KAGGLE_USERNAME = "anbisiadis"

HF_REPO = "allenai/Olmo-3.1-32B-Think"

MODEL_SLUG = "olmo-3-1-32b-think"
FRAMEWORK = "transformers"
VARIATION_SLUG = "default"

KAGGLE_HANDLE = f"{KAGGLE_USERNAME}/{MODEL_SLUG}/{FRAMEWORK}/{VARIATION_SLUG}"
LOCAL_DIR = Path("/content/olmo-3-1-32b-think")

LICENSE_NAME = "Apache 2.0"
VERSION_NOTES = "Mirror of allenai/Olmo-3.1-32B-Think from Hugging Face."

print("Kaggle handle:", KAGGLE_HANDLE)
print("Local staging directory:", LOCAL_DIR)

In [ ]:
print("Disk before download:")
!df -h /content

In [ ]:
from huggingface_hub import snapshot_download

LOCAL_DIR.mkdir(parents=True, exist_ok=True)

snapshot_path = snapshot_download(
    repo_id=HF_REPO,
    repo_type="model",
    local_dir=str(LOCAL_DIR),
    token=HF_TOKEN,
    max_workers=8,
)

print("Downloaded Hugging Face repo to:", snapshot_path)

In [ ]:
import shutil

print("Disk usage before cleanup:")
!du -sh /content/olmo-3-1-32b-think
!df -h /content

shutil.rmtree(LOCAL_DIR / ".cache", ignore_errors=True)

print("\nDisk usage after removing local HF metadata:")
!du -sh /content/olmo-3-1-32b-think
!df -h /content

print("\nTop-level files:")
!find /content/olmo-3-1-32b-think -maxdepth 1 -type f | sort | sed 's|/content/olmo-3-1-32b-think/||' | head -200

print("\nSafetensors files:")
!find /content/olmo-3-1-32b-think -name "*.safetensors" -type f | sort

In [ ]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path
from google.colab import userdata

# Install/upgrade Kaggle CLI.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-U", "kaggle"],
    check=True,
)

KAGGLE_TOKEN = userdata.get("KAGGLE_TOKEN")
if not KAGGLE_TOKEN:
    raise RuntimeError("Missing Colab secret: KAGGLE_TOKEN")

os.environ["KAGGLE_API_TOKEN"] = KAGGLE_TOKEN.strip()

LOCAL_DIR = Path("/content/olmo-3-1-32b-think")
if not LOCAL_DIR.exists():
    raise FileNotFoundError(f"Missing directory: {LOCAL_DIR}")

OWNER_SLUG = "andreasbis"
DATASET_SLUG = "olmo-3-1-32b-think"
DATASET_REF = f"{OWNER_SLUG}/{DATASET_SLUG}"

# Clean wrong/confusing metadata files.
for name in ["dataset-metadata.json", "model-metadata.json"]:
    p = LOCAL_DIR / name
    if p.exists():
        p.unlink()
        print("Deleted:", p)

# Remove Hugging Face local metadata cache, not model files.
shutil.rmtree(LOCAL_DIR / ".cache", ignore_errors=True)

metadata = {
    "title": "OLMo 3.1 32B Think",
    "id": DATASET_REF,
    "licenses": [{"name": "apache-2.0"}],
    "description": "Mirror of allenai/Olmo-3.1-32B-Think from Hugging Face."
}

metadata_path = LOCAL_DIR / "dataset-metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Using dataset metadata:")
print(metadata_path.read_text())

files = [p for p in LOCAL_DIR.rglob("*") if p.is_file()]
safetensors = [p for p in files if p.name.endswith(".safetensors")]

print("\nFolder check:")
print("Directory:", LOCAL_DIR)
print("Total files:", len(files))
print("Safetensors:", len(safetensors))

if len(safetensors) != 14:
    raise RuntimeError(f"Expected 14 safetensors files, found {len(safetensors)}")

print("\nCreating Kaggle Dataset:")
print(f"https://www.kaggle.com/datasets/{DATASET_REF}")

result = subprocess.run(
    [
        "kaggle",
        "datasets",
        "create",
        "-p",
        str(LOCAL_DIR),
        "--dir-mode",
        "skip",
        "--keep-tabular",
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(result.stdout)

lower = result.stdout.lower()
if result.returncode != 0 or "error" in lower or "invalid" in lower or "denied" in lower:
    raise RuntimeError("Dataset creation failed. Read output above.")

print("\nDataset created successfully:")
print(f"https://www.kaggle.com/datasets/{DATASET_REF}")